## 1. Data Loading & Strict Anti-Leakage (Cut-off Implementation)
To simulate a real-world predictive environment and prevent data leakage, a strict `CUT_OFF_DAY` is established.
*   **Logic:** `CUT_OFF_DAY = MAX_DAY - 60`
*   **Transaction Filter:** `obs_txns = transactions[DAY < CUT_OFF_DAY]`. (Strictly `<` to ensure the cut-off day's transactions do not overlap with the churn label window).
*   **Causal Filter:** Filtered using `WEEK_NO <= cutoff_week` to prevent future promotion leakage.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.combine import SMOTETomek
from IPython.display import display

# ==============================================================================
# STEP 1: LOAD DATA & SET CUT-OFF (AIRTIGHT ANTI-LEAKAGE)
# ==============================================================================
DATA_PROCESSED = '../Data/Processed/'
DATA_RAW = '../Data/Raw/'
MODELS_DIR = '../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

print("Loading Data...")
transactions = pd.read_parquet(os.path.join(DATA_PROCESSED, 'transactions_master.parquet'), engine='pyarrow')
customer = pd.read_parquet(os.path.join(DATA_PROCESSED, 'customer_base_labeled.parquet'), engine='pyarrow')
demographics = pd.read_parquet(os.path.join(DATA_PROCESSED, 'demographics_imputed.parquet'), engine='pyarrow')
product = pd.read_parquet(os.path.join(DATA_RAW, 'product.parquet'), engine='pyarrow')
causal_data = pd.read_parquet(os.path.join(DATA_RAW, 'causal_data.parquet'), engine='pyarrow')
campaign_table = pd.read_parquet(os.path.join(DATA_PROCESSED, 'campaign_table_clean.parquet'), engine='pyarrow')
campaign_desc = pd.read_parquet(os.path.join(DATA_PROCESSED, 'campaign_desc_clean.parquet'), engine='pyarrow')

# Data sample
print("\nSample data from transactions_master:")
display(transactions.head())

print("\nSample data from causal_data:")
display(causal_data.head())


print("\nSample data from customer_master:")
display(customer.head())

print("\nSample data from demography:")
display(demographics.head())

# SET CUT-OFF
CUT_OFF_DAY = 711 - 60 
print(f"Cut-off Date: Day {CUT_OFF_DAY}")

# 1.1 Filter transactions (create obs_txns - source for all features)
obs_txns = transactions[transactions['DAY'] < CUT_OFF_DAY].copy()

# 1.2 Filter causal data (fix causal leakage)
# Convert DAY to WEEK_NO (assume 1 week = 7 days)
cutoff_week = int(np.ceil(CUT_OFF_DAY / 7))
causal_obs = causal_data[causal_data['WEEK_NO'] <= cutoff_week].copy()


# 1.3 Initialize final table (keep only ID and churn label, no extra fields)
df_final = customer[['household_key', 'is_churn']].copy()
df_final.rename(columns={'is_churn': 'churn_flag'}, inplace=True)
df_final['household_key'] = df_final['household_key'].astype(str)

Đang nạp dữ liệu...

Sample data from transactions_master:


,household_key,BASKET_ID,DAY,PRODUCT_ID,QUANTITY,SALES_VALUE,STORE_ID,RETAIL_DISC,TRANS_TIME,WEEK_NO,COUPON_DISC,COUPON_MATCH_DISC,DEPARTMENT,BRAND,COMMODITY_DESC
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,PRODUCE,Private,POTATOES
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,PRODUCE,National,ONIONS
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,PRODUCE,Private,VEGETABLES - ALL OTHERS
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,PRODUCE,National,TROPICAL FRUIT
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,PRODUCE,Private,ORGANICS FRUIT & VEGETABLES



Sample data from causal_data:


,PRODUCT_ID,STORE_ID,WEEK_NO,display,mailer
0,26190,286,70,0,A
1,26190,288,70,0,A
2,26190,289,70,0,A
3,26190,292,70,0,A
4,26190,293,70,0,A



Sample data from customer_master:


,household_key,mean_IPT,std_IPT,last_purchase_day,personalized_threshold,recency,is_churn,Frequency,Monetary,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC
0,1,8.506494,4.581494,706,17.669482,5,0,83,4310.160156,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown
1,10,142.750000,159.639959,685,462.029919,26,0,9,234.339996,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
2,100,32.619048,24.499951,691,81.618950,20,0,23,1959.219971,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
3,1000,5.256000,5.900935,706,17.057870,5,0,130,3972.439941,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown,Unknown
4,1001,8.897436,13.774237,710,36.445910,1,0,90,4074.020020,45-54,U,50-74K,Homeowner,Unknown,1,None/Unknown



Sample data from demography:


,AGE_DESC,MARITAL_STATUS_CODE,INCOME_DESC,HOMEOWNER_DESC,HH_COMP_DESC,HOUSEHOLD_SIZE_DESC,KID_CATEGORY_DESC,household_key
0,65+,A,35-49K,Homeowner,2 Adults No Kids,2,None/Unknown,1
1,45-54,A,50-74K,Homeowner,2 Adults No Kids,2,None/Unknown,7
2,25-34,U,25-34K,Unknown,2 Adults Kids,3,1,8
3,25-34,U,75-99K,Homeowner,2 Adults Kids,4,2,13
4,45-54,B,50-74K,Homeowner,Single Female,1,None/Unknown,16


Cut-off Date: Day 651


## 2. RFM (Recency-Safe), Stability & Basket Behavior
Calculates core purchasing habits based purely on the observation window.

| Feature | Logic / Formula | Business Insight |
| :--- | :--- | :--- |
| **Frequency** | Unique `BASKET_ID` count. | Customer's overall store visit frequency. |
| **Monetary** | Sum of `SALES_VALUE`. | Total customer lifetime value (historic). |
| **Avg_Items_Per_Basket** | `Total_Items / Frequency` | Basket depth; indicates if the customer does bulk shopping or quick trips. |
| **Basket_Value_Std** | Standard deviation of basket values. | Measures the consistency of the customer's spending per trip. |
| **Recency_Capped** | `clip(CUT_OFF_DAY - last_day, 0, 90)` | Safe recency metric. Capped at 90 days to reduce noise from long-dormant customers. |
| **Inactive_Days_Ratio** | `Recency_Capped / Customer_Lifetime` | The proportion of the customer's lifespan spent inactive. |
| **active_weeks_ratio** | `Active_Weeks / (Lifetime_in_weeks + 1)` | Engagement depth. Clipped to 1.0 to prevent inflating ratios for new customers. |
| **coupon_dependency** | `Total_Coupon_Discount / Monetary` | Price sensitivity; identifies bargain hunters. |
| **promo_usage_ratio** | `Trips_With_Coupon / Frequency` | Frequency of utilizing promotions during visits. |
| **IPT_mean / IPT_std** | Mean and Std of Inter-Purchase Time (days). | Calculates the average gap between visits and its variance. NaNs are handled to avoid "perfect stability" bias. |
| **IPT_CV** | `IPT_std / IPT_mean` | **Crucial habit metric**. Coefficient of Variation of visit gaps. High CV = erratic shopper; Low CV = creature of habit. |

In [ ]:
# ==============================================================================
# STEP 2: RFM (RECENCY-SAFE), STABILITY (IPT) & BASKET BEHAVIOR
# ==============================================================================
print("Processing RFM, Stability (IPT), and Behavior...")
obs_txns['household_key'] = obs_txns['household_key'].astype(str)

# 2.1 Basic RFM & basket behavior
agg_funcs = {
    'BASKET_ID': ['nunique', 'count'], # nunique = trips, count = total items
    'SALES_VALUE': ['sum', 'std'],     # sum = Monetary, std = basket value std
    'COUPON_DISC': lambda x: np.sum(np.abs(x)), 
    'DAY': ['min', 'max']  
}

cust_feats = obs_txns.groupby('household_key').agg(agg_funcs).reset_index()
cust_feats.columns = ['household_key', 'Frequency', 'Total_Items', 'Monetary', 'Basket_Value_Std', 'Total_Coupon_Discount', 'first_day', 'last_day']

# -------------------------------------------------------------------------
# FIX HERE: move customer_lifetime calculation above
# -------------------------------------------------------------------------
# 2.2 Recency (recency-safe) & lifetime
cust_feats['customer_lifetime'] = CUT_OFF_DAY - cust_feats['first_day']
cust_feats['customer_lifetime'] = np.where(cust_feats['customer_lifetime'] <= 0, 1, cust_feats['customer_lifetime'])

# Capped recency: from last purchase day (in obs_txns) to cut-off
raw_recency = CUT_OFF_DAY - cust_feats['last_day']
# Cap recency at 90 days (reduce noise from very dormant customers)
cust_feats['Recency_Capped'] = np.clip(raw_recency, 0, 90)

# Inactive streak / purchase gap ratio
cust_feats['Inactive_Days_Ratio'] = raw_recency / cust_feats['customer_lifetime']

# -------------------------------------------------------------------------
# CONTINUE OTHER FEATURES
# -------------------------------------------------------------------------
# Active weeks (store visits) / total weeks in customer lifetime
active_weeks = obs_txns.groupby('household_key')['WEEK_NO'].nunique().reset_index(name='Active_Weeks')
cust_feats = pd.merge(cust_feats, active_weeks, on='household_key', how='left')
cust_feats['active_weeks_ratio'] = np.clip(cust_feats['Active_Weeks'] / ((cust_feats['customer_lifetime'] / 7) + 1), 0, 1)

# Trips with coupon / total trips
txns_with_coupon = obs_txns[obs_txns['COUPON_DISC'] < 0].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='Trips_With_Coupon')
cust_feats = pd.merge(cust_feats, txns_with_coupon, on='household_key', how='left').fillna(0)
cust_feats['promo_usage_ratio'] = cust_feats['Trips_With_Coupon'] / cust_feats['Frequency']

# 2.3 Basket & promotion sensitivity
cust_feats['Avg_Items_Per_Basket'] = cust_feats['Total_Items'] / cust_feats['Frequency']
cust_feats['Discount_Per_Transaction'] = cust_feats['Total_Coupon_Discount'] / cust_feats['Frequency']

# Drop unused columns
cust_feats.drop(columns=['Total_Items', 'Total_Coupon_Discount', 'first_day', 'last_day', 'Active_Weeks', 'Trips_With_Coupon'], inplace=True)
cust_feats.fillna(0, inplace=True)

# 2.4 STABILITY / HABIT (IPT - STRONG SIGNAL)
# Get unique purchase days per customer
unique_days = obs_txns[['household_key', 'DAY']].drop_duplicates().sort_values(['household_key', 'DAY'])
unique_days['Prev_DAY'] = unique_days.groupby('household_key')['DAY'].shift(1)
unique_days['IPT'] = unique_days['DAY'] - unique_days['Prev_DAY']

ipt_stats = unique_days.groupby('household_key').agg(
    IPT_mean=('IPT', 'mean'),
    IPT_std=('IPT', 'std')
).reset_index()

# Fix: filling std with 0 can be misleading; use mean for missing std
ipt_stats['IPT_std'] = ipt_stats['IPT_std'].fillna(ipt_stats['IPT_mean'])

# If mean is also NaN (only 1 purchase), fill with customer lifetime
cust_life_map = dict(zip(cust_feats['household_key'], cust_feats['customer_lifetime']))
ipt_stats['IPT_mean'] = ipt_stats['IPT_mean'].fillna(ipt_stats['household_key'].map(cust_life_map))
ipt_stats['IPT_std'] = ipt_stats['IPT_std'].fillna(ipt_stats['household_key'].map(cust_life_map))

# Coefficient of Variation (CV) of IPT
ipt_stats['IPT_CV'] = ipt_stats['IPT_std'] / (ipt_stats['IPT_mean'] + 1e-5)
ipt_stats.fillna(0, inplace=True) # Fill remaining rare cases with 0

# Merge into cust_feats
cust_feats = pd.merge(cust_feats, ipt_stats, on='household_key', how='left').fillna(0)

Đang xử lý RFM, Stability (IPT) và Behavior...


## 3. Marketing Responsiveness, Brand Affinity & Rolling Trend
Captures how the customer interacts with store environments, brands, and recent momentum.

| Feature | Logic / Formula | Business Insight |
| :--- | :--- | :--- |
| **Private_Brand_Ratio** | `(Private + 1) / (Private + National + 2)` | Brand loyalty using **Laplace Smoothing** to prevent NaN bias for single-item buyers. High ratio = high switching cost. |
| **Display / Mailer_Responsiveness** | `Qty_from_Promo / Total_Qty` | Measures how effectively in-store displays and mailers drive the customer's volume. |
| **Rolling_Freq_Slope** | `np.polyfit` over M1, M2, M3 (30-day windows). | Linear regression slope of recent shopping frequency. Captures momentum (cooling down vs. warming up). |

In [ ]:
# ==============================================================================
# STEP 3: MARKETING RESPONSIVENESS, BRAND AFFINITY & ROLLING TREND
# ==============================================================================
print("Processing Brand, Causal, and Rolling Trend...")

# 3.1 Brand affinity (private ratio)
if 'BRAND' not in obs_txns.columns:
    obs_txns = pd.merge(obs_txns, product[['PRODUCT_ID', 'BRAND']], on='PRODUCT_ID', how='left')

brand_qty = obs_txns.groupby(['household_key', 'BRAND'])['QUANTITY'].sum().unstack(fill_value=0).reset_index()
brand_qty['Private_Brand_Ratio'] = (brand_qty.get('Private', 0) + 1) / (brand_qty.get('Private', 0) + brand_qty.get('National', 0) + 2)

# 3.2 Causal data (causal leakage fixed)
causal_obs['PRODUCT_ID'] = causal_obs['PRODUCT_ID'].astype(str)
causal_obs['WEEK_NO'] = causal_obs['WEEK_NO'].astype(int)
obs_txns['PRODUCT_ID'] = obs_txns['PRODUCT_ID'].astype(str)
obs_txns['WEEK_NO'] = obs_txns['WEEK_NO'].astype(int)

causal_obs['is_display'] = np.where(~causal_obs['display'].isin(['0', ' ', 'None', 'NaN']), 1, 0)
causal_obs['is_mailer'] = np.where(~causal_obs['mailer'].isin(['0', ' ', 'None', 'NaN']), 1, 0)
causal_agg = causal_obs.groupby(['WEEK_NO', 'PRODUCT_ID']).agg({'is_display': 'max', 'is_mailer': 'max'}).reset_index()

txn_causal = pd.merge(obs_txns[['household_key', 'WEEK_NO', 'PRODUCT_ID', 'QUANTITY']], causal_agg, on=['WEEK_NO', 'PRODUCT_ID'], how='left').fillna(0)
txn_causal['Qty_Display'] = txn_causal['QUANTITY'] * txn_causal['is_display']
txn_causal['Qty_Mailer'] = txn_causal['QUANTITY'] * txn_causal['is_mailer']

causal_features = txn_causal.groupby('household_key').agg(
    Total_Qty=('QUANTITY', 'sum'),
    Qty_Display=('Qty_Display', 'sum'),
    Qty_Mailer=('Qty_Mailer', 'sum')
).reset_index()

causal_features['Display_Responsiveness'] = causal_features['Qty_Display'] / (causal_features['Total_Qty'] + 1e-5)
causal_features['Mailer_Responsiveness'] = causal_features['Qty_Mailer'] / (causal_features['Total_Qty'] + 1e-5)
causal_features.drop(columns=['Total_Qty', 'Qty_Display', 'Qty_Mailer'], inplace=True)

# 3.3 Rolling trend (linear regression slope over last 3 months)
# Split last 90 days into 3 periods (each 30 days)
obs_txns['days_to_cutoff'] = CUT_OFF_DAY - obs_txns['DAY']

# Monthly frequency
m1 = obs_txns[obs_txns['days_to_cutoff'] <= 30].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='M1')
m2 = obs_txns[(obs_txns['days_to_cutoff'] > 30) & (obs_txns['days_to_cutoff'] <= 60)].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='M2')
m3 = obs_txns[(obs_txns['days_to_cutoff'] > 60) & (obs_txns['days_to_cutoff'] <= 90)].groupby('household_key')['BASKET_ID'].nunique().reset_index(name='M3')

trend_df = pd.DataFrame({'household_key': obs_txns['household_key'].unique()})
trend_df = trend_df.merge(m1, on='household_key', how='left').merge(m2, on='household_key', how='left').merge(m3, on='household_key', how='left').fillna(0)

# Linear slope over 3 points (M3 -> M2 -> M1)
# Simple slope: (y3 - y1) / (x3 - x1). Here x is time, so slope = (M1 - M3) / 2
def calc_slope(row):
    return np.polyfit([1, 2, 3], [row['M3'], row['M2'], row['M1']], 1)[0]

trend_df['Rolling_Freq_Slope'] = trend_df.apply(calc_slope, axis=1)
trend_df = trend_df[['household_key', 'Rolling_Freq_Slope']]

Đang xử lý Brand, Causal và Rolling Trend...


## 4. Campaign Exposure & Demographics
Measures the impact of direct marketing campaigns and profile completeness.

| Feature | Logic / Formula | Business Insight |
| :--- | :--- | :--- |
| **Total_Campaigns_Received** | Count of campaigns before CUT_OFF_DAY. | Overall marketing investment in this customer. |
| **Camp_Count_TypeA/B/C** | Crosstab of campaign types. | Identifies which campaign formats the customer is exposed to. |
| **Campaigns_Last_30D** | Campaigns received in the last 30 days. | **Campaign Fatigue**. Measures if the customer is being spammed right before the churn window. |
| **Days_Since_Last_Camp** | `CUT_OFF_DAY - last_campaign_day`. | Evaluates if the customer is feeling neglected by marketing. (Un-targeted customers imputed with max distance). |
| **has_demographic_info** | `1` if age info exists, else `0`. | Proxy for membership program engagement/completeness. |

In [ ]:
# ==============================================================================
# STEP 4: CAMPAIGN EXPOSURE & HAS_DEMOGRAPHICS
# ==============================================================================
print("Processing Campaign and Demographics...")

# 4.1 Campaign exposure
if 'DESCRIPTION' in campaign_table.columns:
    campaign_table = campaign_table.drop(columns=['DESCRIPTION'])
camp_full = pd.merge(campaign_table, campaign_desc, on='CAMPAIGN', how='left')

# Filter campaigns before cut-off
camp_obs = camp_full[camp_full['START_DAY'] <= CUT_OFF_DAY].copy()
camp_obs['household_key'] = camp_obs['household_key'].astype(str)

# Campaign exposure (total)
camp_counts = camp_obs.groupby('household_key').size().reset_index(name='Total_Campaigns_Received')

# Campaign fatigue (too many in last 30 days)
camp_obs['days_to_cutoff_camp'] = CUT_OFF_DAY - camp_obs['START_DAY']
fatigue = camp_obs[camp_obs['days_to_cutoff_camp'] <= 30].groupby('household_key').size().reset_index(name='Campaigns_Last_30D')

# Campaign recency
camp_recency = camp_obs.groupby('household_key')['START_DAY'].max().reset_index()
camp_recency['Days_Since_Last_Camp'] = CUT_OFF_DAY - camp_recency['START_DAY']

campaign_features = pd.merge(camp_counts, fatigue, on='household_key', how='outer').fillna(0)
campaign_features = pd.merge(campaign_features, camp_recency[['household_key', 'Days_Since_Last_Camp']], on='household_key', how='outer')

# 4.2 Demographics (has_demo)
demographics['household_key'] = demographics['household_key'].astype(str)
demo_feature = pd.DataFrame({'household_key': obs_txns['household_key'].unique()})
demo_feature = pd.merge(demo_feature, demographics[['household_key', 'AGE_DESC']], on='household_key', how='left')
demo_feature['has_demographic_info'] = np.where(demo_feature['AGE_DESC'].notna(), 1, 0)
demo_feature.drop(columns=['AGE_DESC'], inplace=True)

Đang xử lý Campaign và Demographics...


## 5. Assembly, Scaling & SMOTETomek (The Airtight Pipeline)

This final step compiles all engineered features into a single analytical dataset and prepares it for machine learning algorithms. The process follows strict data science protocols to ensure model robustness and prevent data leakage.

**Key Pipeline Mechanisms:**

*   **Feature Consolidation (`pd.merge`):** All individual feature sets are joined together using the `household_key`. Unmatched records (e.g., customers who received no campaigns) are carefully imputed.
*   **Airtight Scaling (`StandardScaler`):** 
    *   *Why?* Machine Learning algorithms (and distance-based techniques like SMOTE) are highly sensitive to the scale of input features. For instance, `Monetary` might be in the thousands, while `Private_Brand_Ratio` is between 0 and 1.
    *   *Anti-Leakage Check:* The scaler is explicitly `.fit()` **ONLY on the Training set**. It is then used to `.transform()` both the Training and Test sets. If the scaler were fit on the entire dataset before splitting, information about the test set's distribution would "leak" into the training phase.
*   **Handling Class Imbalance (`SMOTETomek`):**
    *   *The Problem:* In churn prediction, the "Churners" (Class 1) are a small minority. Without intervention, models will develop a bias toward predicting Class 0 (Retained), resulting in terrible recall for actual churners.
    *   *The Solution:* We use **SMOTETomek**, an advanced hybrid approach:
        1.  **SMOTE (Synthetic Minority Over-sampling Technique):** Creates synthetic, statistically plausible churn customers by finding $k$-nearest neighbors of existing minority instances and interpolating values between them.
        2.  **Tomek Links:** Acts as a data cleaning step. It identifies pairs of instances from opposite classes that are very close to each other (noisy borderline cases) and removes the majority instance. This creates a clearer decision boundary.
    *   *Anti-Leakage Check:* **Crucially, SMOTETomek is applied ONLY to the Training set (`X_train_scaled`).** Generating synthetic samples before the Train/Test split would result in synthetic "twins" bleeding into the Test set, completely invalidating the evaluation metrics.
*   **ID Management (`household_key`):** The `household_key` is dropped before scaling and SMOTE because it is an arbitrary identifier. However, it is carefully preserved and re-attached to the final Test set (`test_final_df`). This allows the downstream team (M5/M6) to map predicted probabilities back to specific real-world customers for targeted interventions.

In [ ]:
# ==============================================================================
# STEP 5 (AIRTIGHT): ASSEMBLY, SCALING & SMOTETomek
# ==============================================================================
print("Assembling final dataset...")

# 5.1 Merge all feature tables
df_final = pd.merge(df_final, cust_feats, on='household_key', how='inner')
df_final = pd.merge(df_final, brand_qty[['household_key', 'Private_Brand_Ratio']], on='household_key', how='left')
df_final = pd.merge(df_final, causal_features, on='household_key', how='left')
df_final = pd.merge(df_final, trend_df, on='household_key', how='left')
df_final = pd.merge(df_final, campaign_features, on='household_key', how='left')
df_final = pd.merge(df_final, demo_feature, on='household_key', how='left')

# Handle NaN (customers with no campaigns)
df_final['Days_Since_Last_Camp'].fillna(CUT_OFF_DAY, inplace=True)
df_final.fillna(0, inplace=True)

# 5.2 EXPORT RAW DATA (FOR M2 CLUSTERING AND EDA)
df_final.to_csv(os.path.join(MODELS_DIR, 'final_ML_features.csv'), index=False)
print(f"Final DataFrame shape: {df_final.shape}")

# 5.3 PREPARE PIPELINE FOR M5
# Drop household_key because it is a string ID that models cannot learn from
X = df_final.drop(columns=['household_key', 'churn_flag'])
y = df_final['churn_flag']

# Split train/test (test size = 20%, stratify to keep churn ratio)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# FIT SCALER ON TRAIN (anti-leakage)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# APPLY SMOTETomek
print("\nBalancing data with SMOTETomek (train set only)...")
smt = SMOTETomek(random_state=42)
X_train_resampled, y_train_resampled = smt.fit_resample(X_train_scaled, y_train)

# Important note for M5
print("*" * 60)
print("NOTICE FOR MEMBER 5 (MACHINE LEARNING):")
print("Train set has been processed with SMOTETomek.")
print("Note: If using Cross-Validation, consider using imblearn.pipeline to wrap SMOTE inside each fold to avoid validation leakage.")
print("*" * 60)

# Rebuild as DataFrames
train_final_df = pd.concat([X_train_resampled, y_train_resampled.reset_index(drop=True)], axis=1)
test_final_df = pd.concat([X_test_scaled, y_test.reset_index(drop=True)], axis=1)

# Re-attach household_key to test set so M5 can map predictions to real customers
test_final_df['household_key'] = df_final.loc[X_test.index, 'household_key'].values

# EXPORT FILES FOR M5
train_final_df.to_parquet(os.path.join(MODELS_DIR, 'final_train_features.parquet'), index=False)
test_final_df.to_parquet(os.path.join(MODELS_DIR, 'final_test_features.parquet'), index=False)

print(f"\nFEATURE ENGINEERING PIPELINE COMPLETE! Files saved to {MODELS_DIR}")

Đang lắp ráp Dữ liệu cuối cùng...
Kích thước DataFrame cuối cùng: (2499, 23)

Đang cân bằng dữ liệu bằng SMOTETomek (Chỉ trên tập Train)...
************************************************************
THÔNG BÁO CHO MEMBER 5 (MACHINE LEARNING):
Tập Train đã được xử lý SMOTETomek.
Lưu ý: Nếu sử dụng Cross-Validation, hãy cân nhắc sử dụng imblearn.pipeline để bọc SMOTE bên trong mỗi Fold nhằm tránh Validation Leakage.
************************************************************

🎉 HOÀN TẤT PIPELINE FEATURE ENGINEERING! Đã lưu các file vào ../models/
